In [121]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np

In [122]:
InputFolder=r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\NAPA\Mandatory\NPI_Requests"

In [123]:
file_list=[]
for (root, dirs, file) in os.walk(InputFolder):
    for f in file:
        if ('.xlsm')in f and  'Updated' in f:
                    file_list.append(f)
file_list

['NPI_Brake Rotor_Request_Updated.xlsm',
 'NPI_MasterCylinder_Request_Updated.xlsm']

In [124]:
file_link=[]

for i in range(len(file_list)):
     for r,d,f in os.walk(InputFolder):
          for files in f:
               if files == file_list[i]:
                    file_link.append(os.path.join(r,files))

In [125]:
len(file_link)

2

In [126]:
file_link[0]

'C:\\Users\\vikram.vadhirajan\\OneDrive - Trico\\Documents - Product Support India Romania\\Catalog\\Score card\\NAPA\\Mandatory\\NPI_Requests\\NPI_Brake Rotor_Request_Updated.xlsm'

In [127]:
cols=["Part Type"]
df_s = pd.DataFrame(columns=cols)
df_s

,Part Type


In [128]:
repl='''C:\\Users\\vikram.vadhirajan\\OneDrive - Trico\\Documents - Product Support India Romania\\Catalog\\Score card\\NAPA\\Mandatory\\NPI_Requests\\'''
IDS=["Filename","Part Type",'Product ID','*\xa0<Name>','<Parent ID>']

In [129]:
for i in range(len(file_link)):
    workbook=openpyxl.load_workbook(file_link[i])
    sheetname=workbook.sheetnames
    print(sheetname)
    for s in sheetname:
        if (("STEP" not in s) and ("Cover" not in s)):
            print(s)
            df=pd.read_excel(file_link[i],sheet_name=s, skiprows=9)
            df["Filename"]=file_link[i].replace(repl,"")
            df["Part Type"]=s
            df=pd.melt(df,id_vars=IDS).reset_index(drop=True)
            df_s=pd.concat([df_s,df])

['Cover', 'STEPChanges', 'Brake Rotor', 'STEPSettings', 'STEPSmartSheetModelInfo', 'STEPLocalization', 'STEPLOV', 'STEPSmartSheetSettings']
Brake Rotor
['Cover', 'STEPChanges', 'Brake Master Cylinder', 'STEPSettings', 'STEPSmartSheetModelInfo', 'STEPLocalization', 'STEPLOV', 'STEPSmartSheetSettings']
Brake Master Cylinder


In [130]:
df_s

,Part Type,Filename,Product ID,* <Name>,<Parent ID>,variable,value
0,Brake Rotor,NPI_Brake Rotor_Request_Updated.xlsm,PRODPCC_136_1376_15•buy_415468146,48880603,Brake Rotor (PRODPCC_136_1376_15),Prop 65?,N
1,Brake Rotor,NPI_Brake Rotor_Request_Updated.xlsm,PRODPCC_136_1376_15•buy_415468154,48880823,Brake Rotor (PRODPCC_136_1376_15),Prop 65?,N
2,Brake Rotor,NPI_Brake Rotor_Request_Updated.xlsm,PRODPCC_136_1376_15•buy_415468176,48880824,Brake Rotor (PRODPCC_136_1376_15),Prop 65?,N
3,Brake Rotor,NPI_Brake Rotor_Request_Updated.xlsm,PRODPCC_136_1376_15•buy_415468168,48880825,Brake Rotor (PRODPCC_136_1376_15),Prop 65?,N
4,Brake Rotor,NPI_Brake Rotor_Request_Updated.xlsm,PRODPCC_136_1376_15•buy_415468065,48880826,Brake Rotor (PRODPCC_136_1376_15),Prop 65?,N
...,...,...,...,...,...,...,...
151,Brake Master Cylinder,NPI_MasterCylinder_Request_Updated.xlsm,PRODPCC_136_1365_15•buy_418148511,M391238,Brake Master Cylinder (PRODPCC_136_1365_15),VMRS Code,13006000
152,Brake Master Cylinder,NPI_MasterCylinder_Request_Updated.xlsm,PRODPCC_136_1365_15•buy_418148503,M391230,Brake Master Cylinder (PRODPCC_136_1365_15),SDS Required,No (135233618)
153,Brake Master Cylinder,NPI_MasterCylinder_Request_Updated.xlsm,PRODPCC_136_1365_15•buy_418148511,M391238,Brake Master Cylinder (PRODPCC_136_1365_15),SDS Required,No (135233618)
154,Brake Master Cylinder,NPI_MasterCylinder_Request_Updated.xlsm,PRODPCC_136_1365_15•buy_418148503,M391230,Brake Master Cylinder (PRODPCC_136_1365_15),Weight.2,NaN


In [131]:
df_Attr=df_s[['Part Type','variable']].drop_duplicates().reset_index(drop=True)
df_Attr

,Part Type,variable
0,Brake Rotor,Prop 65?
1,Brake Rotor,Prop65 Statement
2,Brake Rotor,Product Description
3,Brake Rotor,Reject Reason
4,Brake Rotor,Prop65 Asset Reference ID ☰
...,...,...
140,Brake Master Cylinder,Reservoir Type.2
141,Brake Master Cylinder,Pressure Switch Included
142,Brake Master Cylinder,VMRS Code
143,Brake Master Cylinder,SDS Required


In [132]:
df_MA=pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\NAPA\Mandatory\NPI_Requests\Mandatory_Attributes_List.xlsx",sheet_name='Sheet1')

In [133]:
df_MA

,Part Type,variable,Mandatory?
0,Brake Rotor,Prop 65?,1
1,Brake Rotor,Prop65 Statement,0
2,Brake Rotor,Product Description,0
3,Brake Rotor,Reject Reason,0
4,Brake Rotor,Prop65 Asset Reference ID ☰,0
...,...,...,...
140,Brake Master Cylinder,Reservoir Type,0
141,Brake Master Cylinder,Pressure Switch Included,0
142,Brake Master Cylinder,VMRS Code,1
143,Brake Master Cylinder,SDS Required,1


In [134]:
df_merged=pd.merge(df_s,df_MA,how='left',on=['Part Type','variable'])

In [135]:
df_merged=df_merged[df_merged['Mandatory?'] == 1]

In [136]:
df_merged

,Part Type,Filename,Product ID,* <Name>,<Parent ID>,variable,value,Mandatory?
0,Brake Rotor,NPI_Brake Rotor_Request_Updated.xlsm,PRODPCC_136_1376_15•buy_415468146,48880603,Brake Rotor (PRODPCC_136_1376_15),Prop 65?,N,1.0
1,Brake Rotor,NPI_Brake Rotor_Request_Updated.xlsm,PRODPCC_136_1376_15•buy_415468154,48880823,Brake Rotor (PRODPCC_136_1376_15),Prop 65?,N,1.0
2,Brake Rotor,NPI_Brake Rotor_Request_Updated.xlsm,PRODPCC_136_1376_15•buy_415468176,48880824,Brake Rotor (PRODPCC_136_1376_15),Prop 65?,N,1.0
3,Brake Rotor,NPI_Brake Rotor_Request_Updated.xlsm,PRODPCC_136_1376_15•buy_415468168,48880825,Brake Rotor (PRODPCC_136_1376_15),Prop 65?,N,1.0
4,Brake Rotor,NPI_Brake Rotor_Request_Updated.xlsm,PRODPCC_136_1376_15•buy_415468065,48880826,Brake Rotor (PRODPCC_136_1376_15),Prop 65?,N,1.0
...,...,...,...,...,...,...,...,...
2303,Brake Master Cylinder,NPI_MasterCylinder_Request_Updated.xlsm,PRODPCC_136_1365_15•buy_418148511,M391238,Brake Master Cylinder (PRODPCC_136_1365_15),Fluid Type ☰,DOT,1.0
2382,Brake Master Cylinder,NPI_MasterCylinder_Request_Updated.xlsm,PRODPCC_136_1365_15•buy_418148503,M391230,Brake Master Cylinder (PRODPCC_136_1365_15),VMRS Code,13006000,1.0
2383,Brake Master Cylinder,NPI_MasterCylinder_Request_Updated.xlsm,PRODPCC_136_1365_15•buy_418148511,M391238,Brake Master Cylinder (PRODPCC_136_1365_15),VMRS Code,13006000,1.0
2384,Brake Master Cylinder,NPI_MasterCylinder_Request_Updated.xlsm,PRODPCC_136_1365_15•buy_418148503,M391230,Brake Master Cylinder (PRODPCC_136_1365_15),SDS Required,No (135233618),1.0
